<a href="https://colab.research.google.com/github/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/blob/main/code/The_Food_Hazard_Detection_Challenge_SemEval_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load data

Load training data:

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# download labeled training data 
# load training data:
data = pd.read_csv('data/incidents_train.csv', index_col=0)
trainset, devset = train_test_split(data, test_size=0.2, random_state=2024)

trainset.sample()

,year,month,day,country,title,text,hazard-category,product-category,hazard,product
567,2011,9,15,au,Little Creatures Brewing Pty Ltd—Pale Ale Pint...,PRA No. 2011/12813 Date published 15 Sep 2011 ...,foreign bodies,alcoholic beverages,glass fragment,beer


Load test data from Codalab:

In [2]:
# download testing data (in conception phase the testing data is unlabeled)
# EDIT: the test data now comes labeled, as the competition is over

# load test data:
testset = pd.read_csv('data/incidents_test.csv', index_col=0)

testset.sample()

,year,month,day,country,title,text,hazard-category,product-category,hazard,product
547,2018,6,14,us,Kellogg Company Voluntarily Recalls Honey Smac...,Kellogg Company today announced it is voluntar...,biological,cereals and bakery products,salmonella,breakfast cereals


# Classifictaion

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

text_clf_lr = Pipeline([
    ('vect', TfidfVectorizer(strip_accents='unicode', analyzer='char', ngram_range=(2,5), max_df=0.5, min_df=5)),
     ('clf', LogisticRegression(max_iter=1000)),
    ])

## Evaluation
* On 20% of the training data
* As was suggested by [Randl et al (2023)](https://aclanthology.org/2024.findings-acl.459)

In [4]:
from sklearn.metrics import classification_report, f1_score

for label in ('hazard-category', 'product-category', 'hazard', 'product'):
  print(label.upper())
  text_clf_lr.fit(trainset.title, trainset[label])

  # get development scores:
  devset['predictions-' + label] = text_clf_lr.predict(devset.title)
  print(f'  macro: {f1_score(devset[label], devset["predictions-" + label], zero_division=0, average="macro"):.2f}')
  print(f'  micro: {f1_score(devset[label], devset["predictions-" + label], zero_division=0, average="micro"):.2f}')

  # predict test set:
  testset[label] = text_clf_lr.predict(testset.title)

HAZARD-CATEGORY
  macro: 0.46
  micro: 0.81
PRODUCT-CATEGORY
  macro: 0.39
  micro: 0.66
HAZARD
  macro: 0.14
  micro: 0.54
PRODUCT
  macro: 0.07
  micro: 0.27


In [5]:
def compute_score(hazards_true, products_true, hazards_pred, products_pred):
  # compute f1 for hazards:
  f1_hazards = f1_score(
    hazards_true,
    hazards_pred,
    average='macro'
  )

  # compute f1 for products:
  f1_products = f1_score(
    products_true[hazards_pred == hazards_true],
    products_pred[hazards_pred == hazards_true],
    average='macro'
  )

  return (f1_hazards + f1_products) / 2.

print(f"Score Sub-Task 1: {compute_score(devset['hazard-category'], devset['product-category'], devset['predictions-hazard-category'], devset['predictions-product-category']):.3f}")
print(f"Score Sub-Task 2: {compute_score(devset['hazard'], devset['product'], devset['predictions-hazard'], devset['predictions-product']):.3f}")

Score Sub-Task 1: 0.449
Score Sub-Task 2: 0.121


# Save file for submission

In [6]:
import os
from shutil import make_archive

# save predictions to a new folder:
os.makedirs('./submission/', exist_ok=True)
testset[['hazard-category', 'product-category', 'hazard', 'product']].to_csv('./submission/submission.csv')

# zip the folder (zipfile can be directly uploaded to codalab):
make_archive('./submission', 'zip', './submission')

'c:\\Users\\vaugr\\nlp_proj\\submission.zip'